# TFLite Export – Quantisation & Validation
**Project:** NoteReader – Indian Currency Speaker for the Blind  
**Institute:** RCOEM, Nagpur – CSE Deep Learning Lab  

## What we do here
1. Load the trained MobileNetV2 model from P7.
2. Convert to TFLite using **post-training INT8 quantisation** (representative dataset required).
3. Measure size reduction and accuracy before vs after quantisation.
4. Test the .tflite model in Python to verify it gives correct predictions.
5. Copy the final .tflite, labels.json and model_config.json to a single `/assets/` folder  
   so the React Native app can bundle them.

## Viva Q&A

### Q – What is post-training quantisation?
Standard neural network weights are 32-bit floats (FP32).  
INT8 quantisation maps each weight to an 8-bit integer using a scale factor and zero-point:  
```
int8_val = round(float32_val / scale) + zero_point
```
This reduces model size by ~75 % (4 bytes → 1 byte per weight) and speeds up inference  
on CPUs that support INT8 SIMD (all modern Android devices).

### Q – Why use a representative dataset for calibration?
INT8 quantisation needs to know the **typical range of activations** in each layer  
so it can choose scale factors that minimise accuracy loss.  
We feed ~100 real images through the model during conversion; TFLite records the min/max activation  
per layer and uses them for optimal quantisation.

### Q – What accuracy drop is acceptable?
Typically < 1–2 % on a well-trained model. If the drop is larger, try dynamic-range quantisation  
(weights only, no calibration) which is a safer fallback.

In [ ]:
"""
Load model + test data.
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import json, os, time, shutil

CLASSES = ['10', '20', '50', '100', '200', '500']

# Load final MobileNetV2 model
model = tf.keras.models.load_model('../models/mobilenetv2_final.h5')
print("Model loaded.")
model.summary(line_length=100)

# Load test set
X_test = np.load('../dataset/X_test.npy')
y_test = np.load('../dataset/y_test.npy')

# Apply MobileNetV2 preprocessing: pixel → [-1, 1]
X_test_prep = preprocess_input(X_test.astype('float32'))
y_test_oh   = tf.keras.utils.to_categorical(y_test, 6)

# Baseline accuracy (Keras FP32 model)
_, acc_fp32 = model.evaluate(X_test_prep, y_test_oh, verbose=0)
print(f"\nKeras FP32 model – Test Accuracy: {acc_fp32*100:.2f}%")

In [ ]:
"""
Post-training INT8 quantisation.
Representative dataset: random 100 images from the training set.
"""

X_train = np.load('../dataset/X_train.npy')
X_train_prep = preprocess_input(X_train.astype('float32'))

# Pick 100 representative images for calibration
rng = np.random.default_rng(42)
rep_indices = rng.choice(len(X_train_prep), 100, replace=False)
rep_data = X_train_prep[rep_indices]

def representative_dataset():
    """Generator yielding one image at a time for TFLite calibration."""
    for img in rep_data:
        yield [np.expand_dims(img, 0)]  # shape: (1, 224, 224, 3)

# ── Convert to TFLite INT8 ─────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]              # enable quantisation
converter.representative_dataset = representative_dataset          # calibration data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8  # INT8 input
converter.inference_output_type = tf.int8  # INT8 output

print("Converting to TFLite INT8 (this may take 1-2 minutes)...")
tflite_model_quant = converter.convert()

tflite_path = '../models/currency_detector.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model_quant)

size_fp32_mb = os.path.getsize('../models/mobilenetv2_final.h5') / (1024**2)
size_tflite_mb = os.path.getsize(tflite_path) / (1024**2)
print(f"\n✓ TFLite INT8 model saved: {tflite_path}")
print(f"  FP32 Keras model size:  {size_fp32_mb:.1f} MB")
print(f"  INT8 TFLite model size: {size_tflite_mb:.1f} MB")
print(f"  Size reduction:         {(1 - size_tflite_mb/size_fp32_mb)*100:.1f}%")

In [ ]:
"""
Validate the TFLite INT8 model in Python.
This confirms that the .tflite file gives the same predictions as the Keras model.
"""

def run_tflite_inference(tflite_path, X):
    """Run inference on all images in X using the TFLite interpreter."""
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_scale, input_zero_point = input_details[0]['quantization']

    preds = []
    times = []
    for img in X:
        # Quantise input: float → int8
        img_q = np.expand_dims(img / input_scale + input_zero_point, 0).astype(np.int8)
        interpreter.set_tensor(input_details[0]['index'], img_q)

        start = time.time()
        interpreter.invoke()
        times.append(time.time() - start)

        out = interpreter.get_tensor(output_details[0]['index'])[0]  # int8
        out_scale, out_zero_point = output_details[0]['quantization']
        out_float = (out.astype(np.float32) - out_zero_point) * out_scale  # dequantise
        preds.append(np.argmax(out_float))

    return np.array(preds), np.array(times)

print("Running TFLite inference on test set (may take a minute)...")
y_pred_tflite, infer_times = run_tflite_inference(tflite_path, X_test_prep)

acc_tflite = np.mean(y_pred_tflite == y_test)
avg_ms = np.mean(infer_times) * 1000

print(f"\n✓ TFLite Validation Results:")
print(f"  Test Accuracy (FP32 Keras):  {acc_fp32*100:.2f}%")
print(f"  Test Accuracy (INT8 TFLite): {acc_tflite*100:.2f}%")
print(f"  Accuracy drop:               {(acc_fp32 - acc_tflite)*100:.2f}%")
print(f"  Average inference time:      {avg_ms:.1f} ms / image (Python CPU)")
print(f"  (On Android ARM CPU expect ~50ms; with NNAPI/GPU delegate expect ~15ms)")

# Quantisation comparison table
import pandas as pd
df_q = pd.DataFrame([{
    'Format': 'FP32 Keras .h5',
    'Size (MB)': round(size_fp32_mb, 1),
    'Test Accuracy (%)': round(acc_fp32 * 100, 2),
    'Infer time (ms/img, Python CPU)': 'N/A',
}, {
    'Format': 'INT8 TFLite',
    'Size (MB)': round(size_tflite_mb, 1),
    'Test Accuracy (%)': round(acc_tflite * 100, 2),
    'Infer time (ms/img, Python CPU)': round(avg_ms, 1),
}])
print("\nQuantisation Summary Table:")
print(df_q.to_string(index=False))
df_q.to_csv('../results/quantization_summary.csv', index=False)

In [ ]:
"""
Copy all app-required assets to ../assets/ (the React Native app bundles these).
  currency_detector.tflite
  labels.json
  model_config.json
"""

ASSET_DIR = '../assets'
os.makedirs(ASSET_DIR, exist_ok=True)

shutil.copy(tflite_path, f'{ASSET_DIR}/currency_detector.tflite')
print(f"✓ Copied currency_detector.tflite → {ASSET_DIR}/")

# labels.json is already in ../assets/; verify it exists
labels_path = f'{ASSET_DIR}/labels.json'
if os.path.exists(labels_path):
    print(f"✓ labels.json already present at {labels_path}")
else:
    labels = {str(i): cls for i, cls in enumerate(CLASSES)}
    labels['classes'] = CLASSES
    labels['num_classes'] = len(CLASSES)
    with open(labels_path, 'w') as f:
        json.dump(labels, f, indent=2)
    print(f"✓ labels.json created at {labels_path}")

# model_config.json is already in ../assets/; verify it
cfg_path = f'{ASSET_DIR}/model_config.json'
if os.path.exists(cfg_path):
    print(f"✓ model_config.json already present at {cfg_path}")
    with open(cfg_path) as f:
        cfg = json.load(f)
    print(json.dumps(cfg, indent=2))

print("\n✅ All assets ready for the React Native app.")
print("Next step: copy the /assets folder to NoteReaderApp/assets/model/")